In [5]:
!pip install -q rank-bm25 sentence-transformers faiss-cpu

In [8]:
import os
import re
import time
import json
import pickle
import numpy as np
import pandas as pd
import faiss
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from tqdm.notebook import tqdm
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
from collections import defaultdict

# ── Cấu hình đường dẫn trên Kaggle ────────────────────────────────────────────
# Kaggle tự động mount dataset vào /kaggle/input/
INPUT_DIR = Path("/kaggle/input/datasets/asaniczka/tmdb-movies-dataset-2023-930k-movies")
RAW_CSV = INPUT_DIR / "TMDB_movie_dataset_v11.csv"

# Mọi file xuất ra phải lưu ở /kaggle/working/
OUTPUT_DIR = Path("/kaggle/working/")
CLEANED_CSV = OUTPUT_DIR / "movies_cleaned.csv"
BM25_INDEX = OUTPUT_DIR / "bm25_index.pkl"
FAISS_INDEX = OUTPUT_DIR / "faiss_index.bin"
EMBEDDINGS = OUTPUT_DIR / "embeddings.npy"
MOVIE_META = OUTPUT_DIR / "movie_meta.pkl"

# Cấu hình model
EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
EMBED_DIM = 384
BATCH_SIZE = 512 # Tăng batch_size vì đang dùng GPU

In [24]:
# (Copy nguyên các hàm làm sạch văn bản từ Phase 1 của bạn vào đây)
def clean_text(text: str) -> str:
    if not isinstance(text, str): return ""
    text = text.lower().strip()
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"[^\w\s,]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def clean_list_field(text: str) -> str:
    if not isinstance(text, str) or text.strip() == "": return ""
    text = re.sub(r"[^\w\s,]", " ", text)
    return " ".join([t.strip().lower().replace(" ", "_") for t in text.split(",") if t])

print("Đang đọc dữ liệu...")
df = pd.read_csv(RAW_CSV, low_memory=False)

# Lọc nhanh
df = df[df["status"] == "Released"]
if "adult" in df.columns: df = df[df["adult"] == False]
df = df[df["overview"].notna() & (df["overview"].str.strip() != "")]
df = df[df["title"].notna() & (df["title"].str.strip() != "")]
df = df[df["original_language"] == "en"]
df = df[df["vote_count"] >= 20]
df["release_date"] = pd.to_datetime(df["release_date"], errors="coerce")
df = df[df["release_date"].dt.year >= 1970]

# Làm sạch
print("Đang làm sạch văn bản...")
df["title_clean"] = df["title"].apply(clean_text)
df["overview_clean"] = df["overview"].apply(clean_text)
df["genres_clean"] = df["genres"].apply(clean_list_field)
df["keywords_clean"] = df["keywords"].apply(lambda x: clean_list_field(x) if pd.notna(x) else "")
df["tagline_clean"] = df["tagline"].apply(lambda x: clean_text(x) if pd.notna(x) else "")

# Combined text cho BM25 & Embedding
df["combined_text"] = df.apply(lambda r: " ".join([r["overview_clean"], r["genres_clean"], r["genres_clean"], r["keywords_clean"], r["tagline_clean"]]), axis=1)
df = df[df["combined_text"].str.strip() != ""].reset_index(drop=True)

# Lưu file sạch
cols_to_keep = ["id", "title", "overview", "genres", "keywords", "release_date", "vote_average", "vote_count", "combined_text"]
df_out = df[cols_to_keep].copy()
df_out.to_csv(CLEANED_CSV, index=False)

# Thêm hàm loại bỏ hoàn toàn khoảng trắng và ký tự đặc biệt
def clean_title_compact(text: str) -> str:
    if not isinstance(text, str): return ""
    # Giữ lại chữ và số, viết liền nhau (vd: "Spider-Man 2" -> "spiderman2")
    return re.sub(r"[^\w]", "", text.lower())

print("Đang làm sạch văn bản...")
df["title_clean"] = df["title"].apply(clean_text)
df["title_compact"] = df["title"].apply(clean_title_compact) # Dạng viết liền
df["overview_clean"] = df["overview"].apply(clean_text)
df["genres_clean"] = df["genres"].apply(clean_list_field)
df["keywords_clean"] = df["keywords"].apply(lambda x: clean_list_field(x) if pd.notna(x) else "")
df["tagline_clean"] = df["tagline"].apply(lambda x: clean_text(x) if pd.notna(x) else "")

# Xây dựng combined_text với chiến lược Booting & Compacting
df["combined_text"] = df.apply(lambda r: " ".join([
    r["title_clean"], r["title_clean"], r["title_clean"], r["title_clean"], # Boost Title x4
    r["title_compact"],                                                     # Khớp chuỗi dính liền
    r["overview_clean"], 
    r["genres_clean"], r["genres_clean"],                                   # Boost Genre x2
    r["keywords_clean"], 
    r["tagline_clean"]
]), axis=1)

df = df[df["combined_text"].str.strip() != ""].reset_index(drop=True)

# Cập nhật danh sách cột cần lưu
cols_to_keep = ["id", "title", "overview", "genres", "keywords", "release_date", "vote_average", "vote_count", "combined_text"]
df_out = df[cols_to_keep].copy()
df_out.to_csv(CLEANED_CSV, index=False)
print(f"Hoàn thành Phase 1! Kích thước dataset: {df_out.shape}")


Đang đọc dữ liệu...
Đang làm sạch văn bản...
Đang làm sạch văn bản...
Hoàn thành Phase 1! Kích thước dataset: (25840, 9)


In [25]:
print("--- BUILD BM25 INDEX (PREFIX & WORD LEVEL) ---")

def tokenize_with_prefix(text: str) -> list:
    """Giữ nguyên từ gốc, và thêm prefix 5 ký tự cho từ dài để match một phần."""
    if not isinstance(text, str): return []
    words = text.lower().split()
    tokens = []
    for word in words:
        tokens.append(word)
        # Nếu từ dài từ 6 ký tự trở lên, cắt lấy 5 ký tự đầu làm token phụ
        if len(word) >= 6:
            tokens.append(word[:5])
    return tokens

# Áp dụng Tokenizer mới vào corpus
corpus = [
    tokenize_with_prefix(str(text)) 
    for text in tqdm(df_out["combined_text"], desc="Tokenizing BM25")
]
bm25 = BM25Okapi(corpus)
bm25_index_list = df_out.index.tolist()

with open(BM25_INDEX, "wb") as f:
    pickle.dump({"bm25": bm25, "index": bm25_index_list}, f)



print("\n--- BUILD SEMANTIC INDEX ---")
# Model tự động nhận diện và sử dụng GPU (cuda) của Kaggle
model = SentenceTransformer(EMBED_MODEL)

texts = df_out["combined_text"].tolist()
embeddings = model.encode(
    texts, 
    batch_size=BATCH_SIZE, 
    show_progress_bar=True, 
    normalize_embeddings=True, 
    convert_to_numpy=True
).astype("float32")

# Dùng faiss-cpu để build index
cpu_index = faiss.IndexFlatIP(EMBED_DIM)
cpu_index.add(embeddings)

faiss.write_index(cpu_index, str(FAISS_INDEX))

# Lưu Metadata
meta = df_out.to_dict("records")
with open(MOVIE_META, "wb") as f:
    pickle.dump(meta, f)
    
print("Hoàn thành Phase 2! Đã lưu toàn bộ Index xuống /kaggle/working/")

--- BUILD BM25 INDEX (PREFIX & WORD LEVEL) ---


Tokenizing BM25:   0%|          | 0/25840 [00:00<?, ?it/s]


--- BUILD SEMANTIC INDEX ---


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/51 [00:00<?, ?it/s]

Hoàn thành Phase 2! Đã lưu toàn bộ Index xuống /kaggle/working/


In [27]:
def hybrid_search(query, top_k=5, alpha=0.5):
    # 1. Chuẩn hóa & Mở rộng Query cho BM25
    # Tokenize query bằng hàm prefix
    bm25_query_tokens = tokenize_with_prefix(query)
    
    # Tạo dạng compact của toàn bộ câu query (vd: "spider man" -> "spiderman")
    query_compact = re.sub(r"[^\w]", "", query.lower())
    if query_compact and query_compact not in bm25_query_tokens:
        bm25_query_tokens.append(query_compact)
        
    bm25_scores = bm25.get_scores(bm25_query_tokens)
    bm25_top_ids = np.argsort(bm25_scores)[::-1][:100]
    
    # 2. Semantic Search (giữ nguyên câu query gốc mang ý nghĩa cho model)
    q_vec = model.encode([query], normalize_embeddings=True, convert_to_numpy=True).astype("float32")
    sem_scores, sem_ids = cpu_index.search(q_vec, 100)
    
    # 3. RRF Fusion
    k_rrf = 60
    scores = defaultdict(float)
    
    for rank, row_id in enumerate(bm25_top_ids):
        scores[row_id] += alpha * (1.0 / (k_rrf + rank + 1))
        
    for rank, row_id in enumerate(sem_ids[0]):
        if row_id != -1:
            scores[row_id] += (1 - alpha) * (1.0 / (k_rrf + rank + 1))
            
    # Sort & In kết quả
    merged = sorted([{"row_id": int(rid), "score": s} for rid, s in scores.items()], key=lambda x: x["score"], reverse=True)
    
    print(f"Kết quả cho query: '{query}'\n" + "-"*50)
    for i, res in enumerate(merged[:top_k]):
        m = meta[res["row_id"]]
        print(f"[{i+1}] {m['title']} ({str(m['release_date'])[:4]}) - Score: {res['score']:.4f}")
        print(f"    {str(m['overview'])[:120]}...\n")

# Test thử nghiệm
hybrid_search("spider man") # Test khoảng trắng
hybrid_search("spiderman")  # Test dính liền
hybrid_search("interstellar") # Test sai chính tả

Kết quả cho query: 'spider man'
--------------------------------------------------
[1] Spider-Man (1977) - Score: 0.0158
    When an extortionist threatens to force a multi-suicide unless a huge ransom is paid, only Peter Parker can stop him wit...

[2] Spider-Man 3 (2007) - Score: 0.0154
    The seemingly invincible Spider-Man goes up against an all-new crop of villains—including the shape-shifting Sandman. Wh...

[3] Spider-Man: Into the Spider-Verse (2018) - Score: 0.0152
    Struggling to find his place in the world while juggling school and family, Brooklyn teenager Miles Morales is unexpecte...

[4] Spider-Man: The Venom Saga (1994) - Score: 0.0150
    A space-shuttle crash-landing puts the famous web-slinger Spider-Man in contact with a living alien substance that bonds...

[5] Spider-Man: The Return of the Green Goblin (1997) - Score: 0.0146
    The Webslinger faces the ultimate challenge when his arch-nemesis discovers his identity and kidnaps his one true love, ...

Kết quả c